In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 10
fig_height = 6
fig_format = 'retina'
fig_dpi = 150
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2JlbmphbWlubWEvRGVza3RvcC9xdWFydG9fd2Vic2l0ZS9pbmR1c3RyeS1hcHBsaWNhdGlvbnM='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/_bootstrap.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/_bootstrap_external.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/zipimport.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/codecs.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/encodings/aliases.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/encodings/__init__.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/encodings/utf_8.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/abc.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/io.py": 1728279722.0, "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/stat.py": 1728279722.0, "/Libr

In [2]:
#| echo: false
#| output: false

import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
DATA_PATH = Path("data/amazon_reviews_us_Musical_Instruments_v1_00.tsv").resolve()

df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    on_bad_lines="skip",
    engine="python",
    quoting=3
)

print(f"Loaded {len(df):,} reviews")
print(f"Unique users: {df['customer_id'].nunique():,}")
print(f"Unique products: {df['product_id'].nunique():,}")
print(f"Date range: {df['review_date'].min()} to {df['review_date'].max()}")

df.head()

Loaded 904,765 reviews
Unique users: 573,149
Unique products: 123,328
Date range: 1999-12-13 to 2015-08-31


,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,45610553,RMDCHWD0Y5OZ9,B00HH62VB6,618218723,AGPtek® 10 Isolated Output 9V 12V 18V Guitar P...,Musical Instruments,3,0,1,N,N,Three Stars,"Works very good, but induces ALOT of noise.",2015-08-31
1,US,14640079,RZSL0BALIYUNU,B003LRN53I,986692292,Sennheiser HD203 Closed-Back DJ Headphones,Musical Instruments,5,0,0,N,Y,Five Stars,Nice headphones at a reasonable price.,2015-08-31
2,US,6111003,RIZR67JKUDBI0,B0006VMBHI,603261968,AudioQuest LP record clean brush,Musical Instruments,3,0,1,N,Y,Three Stars,removes dust. does not clean,2015-08-31
3,US,1546619,R27HL570VNL85F,B002B55TRG,575084461,Hohner Inc. 560BX-BF Special Twenty Harmonica,Musical Instruments,5,0,0,N,Y,I purchase these for a friend in return for pl...,I purchase these for a friend in return for pl...,2015-08-31
4,US,12222213,R34EBU9QDWJ1GD,B00N1YPXW2,165236328,Blue Yeti USB Microphone - Blackout Edition,Musical Instruments,5,0,0,N,Y,Five Stars,This is an awesome mic!,2015-08-31


In [4]:
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Rating Distribution ===")
print(df['star_rating'].value_counts().sort_index())

=== Missing Values ===


marketplace           0
customer_id           0
review_id             0
product_id            0
product_parent        0
product_title         1
product_category      0
star_rating           0
helpful_votes         0
total_votes           0
vine                  0
verified_purchase     0
review_headline       6
review_body          63
review_date           0
dtype: int64

=== Data Types ===
marketplace          object
customer_id           int64
review_id            object
product_id           object
product_parent        int64
product_title        object
product_category     object
star_rating           int64
helpful_votes         int64
total_votes           int64
vine                 object
verified_purchase    object
review_headline      object
review_body          object
review_date          object
dtype: object

=== Rating Distribution ===
star_rating
1     66141
2     40130
3     67045
4    158532
5    572917
Name: count, dtype: int64


In [5]:
MIN_USER_REVIEWS = 5
user_counts = df['customer_id'].value_counts()
active_users = user_counts[user_counts >= MIN_USER_REVIEWS].index

df_active = df[df['customer_id'].isin(active_users)].copy()

print(f"Filtered from {len(df):,} to {len(df_active):,} reviews")
print(f"Active users: {len(active_users):,} (≥{MIN_USER_REVIEWS} reviews)")
print(f"Retention rate: {len(df_active)/len(df)*100:.1f}%")

Filtered from 904,765 to 193,996 reviews
Active users: 22,132 (≥5 reviews)
Retention rate: 21.4%


In [6]:
# Aggregate user-level statistics
user_profile = df_active.groupby('customer_id').agg({
    'star_rating': ['mean', 'std', 'min', 'max', 'count'],
    'verified_purchase': lambda x: (x == 'Y').mean(),
    'helpful_votes': ['sum', 'mean'],
    'review_date': ['min', 'max']
}).reset_index()

# Flatten column names
user_profile.columns = ['_'.join(col).strip('_') for col in user_profile.columns.values]
user_profile.columns = [
    'customer_id', 'avg_rating', 'rating_std', 'min_rating', 'max_rating',
    'review_count', 'verified_ratio', 'total_helpful_votes', 'avg_helpful_votes',
    'first_review_date', 'last_review_date'
]

# Derived features
user_profile['rating_range'] = user_profile['max_rating'] - user_profile['min_rating']
user_profile['account_age_days'] = (
    pd.to_datetime(user_profile['last_review_date']) -
    pd.to_datetime(user_profile['first_review_date'])
).dt.days
user_profile['review_frequency'] = user_profile['review_count'] / (user_profile['account_age_days'] + 1)

print("=== User Profile Statistics ===")
print(user_profile.describe().round(2))

user_profile.head(10)

=== User Profile Statistics ===
       customer_id  avg_rating  rating_std  min_rating  max_rating  \
count     22132.00    22132.00    22132.00    22132.00    22132.00   
mean   28329169.65        4.40        0.76        3.04        4.97   
std    15242420.72        0.57        0.56        1.42        0.23   
min       10987.00        1.00        0.00        1.00        1.00   
25%    14844951.00        4.12        0.38        2.00        5.00   
50%    27112330.50        4.50        0.74        3.00        5.00   
75%    42535244.75        4.83        1.17        4.00        5.00   
max    53095826.00        5.00        2.19        5.00        5.00   

       review_count  verified_ratio  total_helpful_votes  avg_helpful_votes  \
count      22132.00        22132.00             22132.00           22132.00   
mean           8.77            0.86                16.61               1.79   
std            8.03            0.23                53.64               4.74   
min            5.00  

,customer_id,avg_rating,rating_std,min_rating,max_rating,review_count,verified_ratio,total_helpful_votes,avg_helpful_votes,first_review_date,last_review_date,rating_range,account_age_days,review_frequency
0,10987,4.625000,0.517549,4,5,8,1.000000,17,2.125000,2013-01-03,2015-05-01,1,848,0.009423
1,13397,4.571429,0.786796,3,5,7,0.714286,2,0.285714,2014-05-14,2014-07-04,2,51,0.134615
2,20928,5.000000,0.000000,5,5,6,1.000000,0,0.000000,2015-01-12,2015-06-23,0,162,0.036810
3,27403,4.000000,0.000000,4,4,5,1.000000,1,0.200000,2014-02-05,2014-09-17,0,224,0.022222
4,29173,5.000000,0.000000,5,5,7,1.000000,1,0.142857,2013-06-12,2013-12-06,0,177,0.039326
5,34910,4.700000,0.483046,4,5,10,0.500000,14,1.400000,2014-03-05,2015-01-19,1,320,0.031153
6,37123,5.000000,0.000000,5,5,5,0.800000,3,0.600000,2014-11-18,2015-03-25,0,127,0.039062
7,38266,4.090909,0.943880,2,5,11,0.909091,4,0.363636,2013-07-06,2015-08-15,3,770,0.014267
8,39575,4.111111,0.781736,3,5,9,1.000000,12,1.333333,2012-11-16,2015-02-24,2,830,0.010830
9,39749,2.600000,1.673320,1,5,5,1.000000,5,1.000000,2014-10-13,2015-07-17,4,277,0.017986


In [7]:
# Add text-based features
df_active['review_length'] = df_active['review_body'].astype(str).apply(len)
df_active['word_count'] = df_active['review_body'].astype(str).apply(lambda x: len(x.split()))
df_active['uppercase_ratio'] = df_active['review_body'].astype(str).apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)
)

# Aggregate text features per user
user_text_features = df_active.groupby('customer_id').agg({
    'review_length': ['mean', 'std'],
    'word_count': ['mean', 'std'],
    'uppercase_ratio': 'mean'
}).reset_index()

user_text_features.columns = ['_'.join(col).strip('_') for col in user_text_features.columns.values]
user_text_features.columns = [
    'customer_id', 'avg_review_length', 'std_review_length',
    'avg_word_count', 'std_word_count', 'avg_uppercase_ratio'
]

# Merge with user profile
user_profile = user_profile.merge(user_text_features, on='customer_id', how='left')

print("\n=== Review Quality Features ===")
print(user_profile[['avg_review_length', 'avg_word_count', 'avg_uppercase_ratio']].describe().round(2))


=== Review Quality Features ===
       avg_review_length  avg_word_count  avg_uppercase_ratio
count           22132.00        22132.00             22132.00
mean              404.95           74.53                 0.04
std               514.10           92.11                 0.06
min                 1.00            1.00                 0.00
25%               129.04           24.20                 0.02
50%               256.53           48.20                 0.03
75%               490.41           91.20                 0.04
max             13633.29         2505.43                 0.92


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select features for clustering
cluster_features = [
    'avg_rating', 'rating_std', 'review_count', 'verified_ratio',
    'avg_helpful_votes', 'avg_review_length', 'review_frequency'
]

X_cluster = user_profile[cluster_features].fillna(0)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# K-Means clustering (elbow method suggests 4-5 clusters)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
user_profile['user_segment'] = kmeans.fit_predict(X_scaled)

# Analyze segments
print("\n=== User Segments Analysis ===")
segment_summary = user_profile.groupby('user_segment')[cluster_features].mean().round(2)
segment_counts = user_profile['user_segment'].value_counts().sort_index()

for seg in range(4):
    print(f"\n Segment {seg} (n={segment_counts[seg]:,}, {segment_counts[seg]/len(user_profile)*100:.1f}%)")
    print(segment_summary.loc[seg])


=== User Segments Analysis ===

 Segment 0 (n=2,227, 10.1%)
avg_rating              4.29
rating_std              0.81
review_count           10.94
verified_ratio          0.40
avg_helpful_votes       7.68
avg_review_length    1350.35
review_frequency        0.16
Name: 0, dtype: float64

 Segment 1 (n=7,064, 31.9%)
avg_rating             3.86
rating_std             1.34
review_count           8.44
verified_ratio         0.89
avg_helpful_votes      1.46
avg_review_length    368.07
review_frequency       0.18
Name: 1, dtype: float64

 Segment 2 (n=10,513, 47.5%)
avg_rating             4.74
rating_std             0.42
review_count           8.96
verified_ratio         0.93
avg_helpful_votes      1.06
avg_review_length    288.11
review_frequency       0.09
Name: 2, dtype: float64

 Segment 3 (n=2,328, 10.5%)
avg_rating             4.62
rating_std             0.47
review_count           6.77
verified_ratio         0.94
avg_helpful_votes      0.42
avg_review_length    140.07
review_frequency

In [9]:
#| label: fig-user-segments
#| fig-cap: User Segmentation Analysis

import matplotlib.pyplot as plt
import numpy as np

# Create figure
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot 1: Avg Rating vs Review Count (USER-LEVEL)
scatter1 = axes[0, 0].scatter(
    user_profile['review_count'],
    user_profile['avg_rating'],
    c=user_profile['user_segment'],
    cmap='viridis',
    alpha=0.6,
    s=30,
    edgecolors='black',
    linewidth=0.5
)
axes[0, 0].set_xlabel('Review Count (log scale)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Average Rating', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Review Activity vs Rating Tendency', fontsize=12, fontweight='bold')
axes[0, 0].set_xscale('log')
axes[0, 0].set_ylim(0.5, 5.5)
axes[0, 0].grid(alpha=0.3, linestyle='--')
cbar1 = plt.colorbar(scatter1, ax=axes[0, 0])
cbar1.set_label('Segment', fontsize=10)

# Plot 2: Rating Std vs Verified Ratio (USER-LEVEL)
scatter2 = axes[0, 1].scatter(
    user_profile['verified_ratio'],
    user_profile['rating_std'],
    c=user_profile['user_segment'],
    cmap='viridis',
    alpha=0.6,
    s=30,
    edgecolors='black',
    linewidth=0.5
)
axes[0, 1].set_xlabel('Verified Purchase Ratio', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Rating Std Dev', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Purchase Verification vs Rating Consistency', fontsize=12, fontweight='bold')
axes[0, 1].set_xlim(-0.05, 1.05)
axes[0, 1].grid(alpha=0.3, linestyle='--')
cbar2 = plt.colorbar(scatter2, ax=axes[0, 1])
cbar2.set_label('Segment', fontsize=10)

# Plot 3: Segment Distribution
colors_bar = plt.cm.viridis(np.linspace(0, 1, 4))
segment_counts.plot(kind='bar', ax=axes[1, 0], color=colors_bar, edgecolor='black', linewidth=1.2)
axes[1, 0].set_xlabel('Segment', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('User Count', fontsize=11, fontweight='bold')
axes[1, 0].set_title('User Segment Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)
axes[1, 0].grid(axis='y', alpha=0.3, linestyle='--')

for i, (idx, count) in enumerate(segment_counts.items()):
    axes[1, 0].text(i, count + 200, f'{count:,}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 4: Helpful Votes vs Review Length (USER-LEVEL)
user_profile_plot = user_profile[
    (user_profile['avg_review_length'] > 0) &
    (user_profile['avg_helpful_votes'] >= 0)
]

scatter4 = axes[1, 1].scatter(
    user_profile_plot['avg_review_length'],
    user_profile_plot['avg_helpful_votes'],
    c=user_profile_plot['user_segment'],
    cmap='viridis',
    alpha=0.6,
    s=30,
    edgecolors='black',
    linewidth=0.5
)
axes[1, 1].set_xlabel('Avg Review Length (chars, log scale)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Avg Helpful Votes', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Review Quality Indicators', fontsize=12, fontweight='bold')
axes[1, 1].set_xscale('log')
axes[1, 1].set_yscale('symlog', linthresh=1)
axes[1, 1].grid(alpha=0.3, linestyle='--')
cbar4 = plt.colorbar(scatter4, ax=axes[1, 1])
cbar4.set_label('Segment', fontsize=10)

plt.tight_layout()

# CRITICAL: Display the figure
plt.show()

In [10]:
# Aggregate product-level features
item_stats = df_active.groupby('product_id').agg({
    'star_rating': ['mean', 'std', 'count'],
    'review_date': ['min', 'max'],
    'verified_purchase': lambda x: (x == 'Y').mean(),
    'product_title': 'first'  # Assume consistent within product_id
}).reset_index()

item_stats.columns = ['_'.join(col).strip('_') for col in item_stats.columns.values]
item_stats.columns = [
    'product_id', 'item_avg_rating', 'item_rating_std', 'item_review_count',
    'item_first_review', 'item_last_review', 'item_verified_ratio', 'product_title'
]

item_stats['item_age_days'] = (
    pd.to_datetime(item_stats['item_last_review']) -
    pd.to_datetime(item_stats['item_first_review'])
).dt.days

print("=== Product Statistics ===")
print(item_stats.describe().round(2))

item_stats.head(10)

=== Product Statistics ===
       item_avg_rating  item_rating_std  item_review_count  \
count         57568.00         24246.00           57568.00   
mean              4.35             0.71               3.37   
std               0.96             0.67              10.05   
min               1.00             0.00               1.00   
25%               4.00             0.00               1.00   
50%               4.92             0.67               1.00   
75%               5.00             1.14               3.00   
max               5.00             2.83             703.00   

       item_verified_ratio  item_age_days  
count             57568.00       57568.00  
mean                  0.80         317.21  
std                   0.36         560.12  
min                   0.00           0.00  
25%                   0.75           0.00  
50%                   1.00           0.00  
75%                   1.00         452.00  
max                   1.00        4872.00  


,product_id,item_avg_rating,item_rating_std,item_review_count,item_first_review,item_last_review,item_verified_ratio,product_title,item_age_days
0,0046197141,5.0,NaN,1,2015-04-18,2015-04-18,1.0,Aria for Alto Saxophone and Piano,0
1,0634029363,5.0,0.000000,2,2013-07-05,2015-06-01,0.5,Hal Leonard Eric Clapton - Acoustic Classics (...,696
2,0739075934,5.0,NaN,1,2015-03-11,2015-03-11,0.0,Alfred BILLY MARTINS LIFE ON DRUMS DVD,0
3,0757918476,5.0,NaN,1,2010-07-23,2010-07-23,1.0,Alfred Conversations in Clave (DVD),0
4,0786615303,3.0,1.414214,2,2015-01-12,2015-02-11,1.0,Premium Clarke Tinwhistle-Key of C,30
5,0793564018,4.0,NaN,1,2008-09-16,2008-09-16,1.0,Love Songs Of The 90s - E-z Play Today,0
6,0825856655,3.0,NaN,1,2015-03-25,2015-03-25,1.0,Carl Fischer The New Extended Working Range fo...,0
7,0879306211,4.8,0.447214,5,2004-10-21,2015-04-25,0.6,Hal Leonard 50 Years of Fender - Half a Centur...,3838
8,0895246074,5.0,NaN,1,2015-07-21,2015-07-21,1.0,Van Halen Bass & Vocal with Tablatire - Cherry...,0
9,0966502949,5.0,NaN,1,2014-08-15,2014-08-15,1.0,Guitar from Scratch - A Cordial Introduction f...,0


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# For demonstration, sample 500 products
np.random.seed(42)
sample_items = item_stats.sample(n=min(500, len(item_stats)), random_state=42)

# Text similarity using TF-IDF
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = tfidf.fit_transform(sample_items['product_title'].fillna(''))

# Compute pairwise cosine similarity
similarity_matrix = cosine_similarity(tfidf_matrix)

# Find high-similarity pairs (>0.8 threshold, excluding self-matches)
potential_duplicates = []
threshold = 0.8

for i in range(len(similarity_matrix)):
    for j in range(i+1, len(similarity_matrix)):
        if similarity_matrix[i, j] > threshold:
            potential_duplicates.append({
                'product_id_1': sample_items.iloc[i]['product_id'],
                'product_id_2': sample_items.iloc[j]['product_id'],
                'text_similarity': similarity_matrix[i, j],
                'rating_diff': abs(sample_items.iloc[i]['item_avg_rating'] -
                                 sample_items.iloc[j]['item_avg_rating']),
                'count_ratio': min(sample_items.iloc[i]['item_review_count'],
                                 sample_items.iloc[j]['item_review_count']) /
                             max(sample_items.iloc[i]['item_review_count'],
                                 sample_items.iloc[j]['item_review_count'])
            })

duplicates_df = pd.DataFrame(potential_duplicates)

print(f"\n=== Potential Duplicate Pairs ===")
print(f"Found {len(duplicates_df)} candidate pairs (similarity >{threshold})")
if len(duplicates_df) > 0:
    print("\nExample duplicate candidates:")
    print(duplicates_df.head(10))


=== Potential Duplicate Pairs ===
Found 197 candidate pairs (similarity >0.8)

Example duplicate candidates:
  product_id_1 product_id_2  text_similarity  rating_diff  count_ratio
0   B004Z2NTW0   B0051UUIBK         1.000000     0.600000     0.200000
1   B004Z2NTW0   B002024UFC         1.000000     0.400000     0.100000
2   B004Z2NTW0   B0051UUI2Y         1.000000     0.000000     0.333333
3   B004Z2NTW0   B004Z2NTCK         1.000000     1.000000     1.000000
4   B0018TJBRK   B001NMT6YK         1.000000     3.000000     1.000000
5   B0006ZRO8A   B001RMC0IK         1.000000     2.333333     0.666667
6   B004W1PQM0   B006LI56EU         0.872998     1.000000     1.000000
7   B004W1PQM0   B003ES5HS0         1.000000     1.000000     0.500000
8   B0002D0DS4   B0002D0DSO         1.000000     0.666667     0.666667
9   B000TOF29Q   B000TO95P8         0.910469     0.400000     0.400000


In [12]:
# Decision rule: Match if text_similarity > 0.85 AND rating_diff < 0.5
if len(duplicates_df) > 0:
    duplicates_df['is_match'] = (
        (duplicates_df['text_similarity'] > 0.85) &
        (duplicates_df['rating_diff'] < 0.5)
    )

    print(f"\n=== Matching Results ===")
    print(f"Confirmed matches: {duplicates_df['is_match'].sum()}")
    print(f"False positives (rejected): {(~duplicates_df['is_match']).sum()}")

    # Show confirmed matches
    if duplicates_df['is_match'].any():
        print("\nConfirmed duplicate pairs:")
        print(duplicates_df[duplicates_df['is_match']][
            ['product_id_1', 'product_id_2', 'text_similarity', 'rating_diff']
        ].head(10))
else:
    print("No potential duplicates found in sample. Try larger sample or lower threshold.")


=== Matching Results ===
Confirmed matches: 62
False positives (rejected): 135

Confirmed duplicate pairs:
   product_id_1 product_id_2  text_similarity  rating_diff
1    B004Z2NTW0   B002024UFC         1.000000     0.400000
2    B004Z2NTW0   B0051UUI2Y         1.000000     0.000000
9    B000TOF29Q   B000TO95P8         0.910469     0.400000
13   B00I0283NG   B0068Y7ODI         0.952154     0.083333
14   B004T9ZF12   B000EEJ6T8         1.000000     0.333333
15   B004T9ZF12   B005N923L2         1.000000     0.333333
19   B00DHO5YY8   B0002D0Q90         1.000000     0.000000
20   B00DHO5YY8   B00BEMM0Z6         0.859610     0.000000
22   B002CG68FQ   B001G7UWIQ         1.000000     0.285714
23   B002CG68FQ   B001TFIN36         1.000000     0.000000


In [13]:
# Precision: % of predicted matches that are true duplicates
# Recall: % of true duplicates that are detected
# F1-Score: Harmonic mean of precision and recall

from sklearn.metrics import precision_score, recall_score, f1_score

# If you had ground truth:
# precision = precision_score(y_true, y_pred)
# recall = recall_score(y_true, y_pred)
# f1 = f1_score(y_true, y_pred)

In [14]:
# Basic text features
df_active['review_length'] = df_active['review_body'].astype(str).apply(len)
df_active['word_count'] = df_active['review_body'].astype(str).apply(lambda x: len(x.split()))
df_active['sentence_count'] = df_active['review_body'].astype(str).apply(lambda x: len(x.split('.')))
df_active['avg_word_length'] = df_active['review_body'].astype(str).apply(
    lambda x: np.mean([len(word) for word in x.split()] + [0])
)

# Style features
df_active['uppercase_ratio'] = df_active['review_body'].astype(str).apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)
)
df_active['punctuation_count'] = df_active['review_body'].astype(str).apply(
    lambda x: sum(1 for c in x if c in '!?.')
)
df_active['exclamation_count'] = df_active['review_body'].astype(str).apply(
    lambda x: x.count('!')
)

print("=== Text Features Summary ===")
print(df_active[['word_count', 'sentence_count', 'avg_word_length',
                 'uppercase_ratio', 'punctuation_count']].describe().round(2))

=== Text Features Summary ===
       word_count  sentence_count  avg_word_length  uppercase_ratio  \
count   193996.00       193996.00        193996.00        193996.00   
mean        81.24            6.66             4.23             0.04   
std        145.54            9.51             0.76             0.08   
min          1.00            1.00             0.50             0.00   
25%         19.00            2.00             3.94             0.02   
50%         39.00            4.00             4.25             0.02   
75%         90.00            7.00             4.54             0.04   
max       8826.00          542.00            43.50             0.92   

       punctuation_count  
count          193996.00  
mean                6.22  
std                10.07  
min                 0.00  
25%                 2.00  
50%                 4.00  
75%                 7.00  
max               584.00  


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Sample for demonstration
sample_df = df_active.sample(n=min(5000, len(df_active)), random_state=42)

# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(
    max_features=50,  # Top 50 terms
    stop_words='english',
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=5  # Term must appear in at least 5 documents
)

tfidf_matrix = tfidf_vectorizer.fit_transform(sample_df['review_body'].fillna(''))
tfidf_features = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{term}' for term in tfidf_vectorizer.get_feature_names_out()]
)

print(f"\n=== TF-IDF Features ===")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Feature matrix shape: {tfidf_matrix.shape}")
print("\nTop terms by average TF-IDF:")
print(tfidf_features.mean().sort_values(ascending=False).head(10))


=== TF-IDF Features ===
Vocabulary size: 50
Feature matrix shape: (5000, 50)

Top terms by average TF-IDF:
tfidf_br        0.103939
tfidf_great     0.103576
tfidf_good      0.089153
tfidf_guitar    0.067946
tfidf_sound     0.062148
tfidf_like      0.061644
tfidf_use       0.056520
tfidf_just      0.056427
tfidf_price     0.052342
tfidf_works     0.051593
dtype: float64


In [16]:
# Binary classification: High rating (≥4) vs Low rating (<4)
df_active['label'] = (df_active['star_rating'] >= 4).astype(int)

print("=== Label Distribution ===")
print(df_active['label'].value_counts())
print(f"\nPositive rate: {df_active['label'].mean()*100:.1f}%")

=== Label Distribution ===
label
1    165274
0     28722
Name: count, dtype: int64

Positive rate: 85.2%


In [17]:
from sklearn.model_selection import train_test_split

# Temporal split for realistic evaluation
df_sorted = df_active.sort_values('review_date')
split_idx = int(len(df_sorted) * 0.8)

train_df = df_sorted.iloc[:split_idx].copy()
test_df = df_sorted.iloc[split_idx:].copy()

print(f"Train: {len(train_df):,} reviews ({train_df['label'].mean()*100:.1f}% positive)")
print(f"Test: {len(test_df):,} reviews ({test_df['label'].mean()*100:.1f}% positive)")

Train: 155,196 reviews (84.9% positive)
Test: 38,800 reviews (86.3% positive)


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Select features
text_features = ['word_count', 'sentence_count', 'avg_word_length',
                'uppercase_ratio', 'punctuation_count', 'exclamation_count']

X_train = train_df[text_features].fillna(0)
X_test = test_df[text_features].fillna(0)
y_train = train_df['label']
y_test = test_df['label']

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predict
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluate
auc_lr = roc_auc_score(y_test, y_pred_proba_lr)

print("=== Logistic Regression Results ===")
print(f"AUC: {auc_lr:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Low Rating', 'High Rating']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

=== Logistic Regression Results ===
AUC: 0.6222

Classification Report:
              precision    recall  f1-score   support

  Low Rating       0.30      0.00      0.00      5333
 High Rating       0.86      1.00      0.93     33467

    accuracy                           0.86     38800
   macro avg       0.58      0.50      0.46     38800
weighted avg       0.78      0.86      0.80     38800


Confusion Matrix:
[[    8  5325]
 [   19 33448]]


In [19]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report

# Build DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# XGBoost parameters
params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "max_depth": 6,
    "eta": 0.1,
    "verbosity": 0,
}

# Train model
bst = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=50
)

# Predictions
y_proba_xgb = bst.predict(dtest)
y_pred_xgb = (y_proba_xgb >= 0.5).astype(int)

# Evaluation
auc_xgb = roc_auc_score(y_test, y_proba_xgb)

print("=== XGBoost Results ===")
print(f"AUC: {auc_xgb:.4f}")
print(f"Improvement over LR: +{auc_xgb - auc_lr:.4f}")
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=["Low Rating", "High Rating"],
        zero_division=0
    )
)

=== XGBoost Results ===
AUC: 0.6665
Improvement over LR: +0.0443

Classification Report:
              precision    recall  f1-score   support

  Low Rating       0.00      0.00      0.00      5333
 High Rating       0.86      1.00      0.93     33467

    accuracy                           0.86     38800
   macro avg       0.43      0.50      0.46     38800
weighted avg       0.74      0.86      0.80     38800



In [20]:
### 4.8 Feature Importance

#| fig-width: 10
#| fig-height: 6

# XGBoost feature importance - diagnostics first
print("=== Feature Importance Extraction ===")

# Get importance scores (try gain first, fall back to weight)
importance_dict = bst.get_score(importance_type='gain')
if len(importance_dict) == 0:
    print(" No 'gain' scores found, using 'weight' instead")
    importance_dict = bst.get_score(importance_type='weight')

print(f"Found {len(importance_dict)} features with scores")
print(f"Feature names from model: {list(importance_dict.keys())}")

# XGBoost uses internal names like 'f0', 'f1', etc.
# But sometimes uses actual feature names - need to handle both
if any(key.startswith('f') and key[1:].isdigit() for key in importance_dict.keys()):
    # Internal naming (f0, f1, ...)
    print("Using internal feature indexing (f0, f1, ...)")
    importance_df = pd.DataFrame({
        'feature': text_features,
        'importance': [importance_dict.get(f'f{i}', 0) for i in range(len(text_features))]
    })
else:
    # Direct feature names
    print("Using direct feature names")
    importance_df = pd.DataFrame({
        'feature': text_features,
        'importance': [importance_dict.get(feat, 0) for feat in text_features]
    })

importance_df = importance_df.sort_values('importance', ascending=False)

# Normalize for better visualization
if importance_df['importance'].sum() > 0:
    importance_df['importance_pct'] = (
        importance_df['importance'] / importance_df['importance'].sum() * 100
    )
else:
    importance_df['importance_pct'] = 0

print("\n=== Feature Importance Table ===")
print(importance_df.round(4))

# Plot
if importance_df['importance'].sum() > 0:
    fig, ax = plt.subplots(figsize=(10, 6))

    # Create horizontal bar chart
    bars = ax.barh(
        importance_df['feature'],
        importance_df['importance_pct'],
        color='steelblue',
        edgecolor='navy',
        linewidth=1.5
    )

    # Styling
    ax.set_xlabel('Relative Importance (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
    ax.set_title('Feature Importance for Review Sentiment Prediction',
                 fontsize=14, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.7)
    ax.invert_yaxis()

    # Add percentage labels on bars
    for bar, val in zip(bars, importance_df['importance_pct']):
        if val > 0:
            ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                   f'{val:.1f}%',
                   va='center', ha='left', fontsize=10, fontweight='bold')

    # Add subtle background color for top feature
    top_feature_idx = 0
    bars[top_feature_idx].set_color('coral')
    bars[top_feature_idx].set_edgecolor('darkred')

    plt.tight_layout()
    plt.show()

    # Key insights
    print(f"\n Key Insights:")
    print(f"   • Top feature: {importance_df.iloc[0]['feature']} ({importance_df.iloc[0]['importance_pct']:.1f}%)")
    print(f"   • Features used: {(importance_df['importance'] > 0).sum()}/{len(text_features)}")

    top3 = importance_df.head(3)
    print(f"   • Top 3 features account for {top3['importance_pct'].sum():.1f}% of total importance")

else:
    print("\n Warning: All importances are zero!")
    print("Possible reasons:")
    print("1. Features have no predictive power")
    print("2. num_boost_round too low (try 100+)")
    print("3. Features are perfectly correlated")
    print("4. Data leakage or constant predictions")

    # Diagnostic plot - show feature variance
    fig, ax = plt.subplots(figsize=(10, 6))
    feature_std = X_train[text_features].std()
    ax.barh(text_features, feature_std, color='coral')
    ax.set_xlabel('Standard Deviation')
    ax.set_title('Feature Variance (Diagnostic)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

=== Feature Importance Extraction ===
Found 6 features with scores
Feature names from model: ['word_count', 'sentence_count', 'avg_word_length', 'uppercase_ratio', 'punctuation_count', 'exclamation_count']
Using direct feature names

=== Feature Importance Table ===
             feature  importance  importance_pct
5  exclamation_count     43.3943         43.7907
0         word_count     25.2040         25.4343
4  punctuation_count     11.9378         12.0468
2    avg_word_length      7.3497          7.4169
1     sentence_count      5.6367          5.6881
3    uppercase_ratio      5.5723          5.6232



 Key Insights:
   • Top feature: exclamation_count (43.8%)
   • Features used: 6/6
   • Top 3 features account for 81.3% of total importance


In [21]:
def compute_ranking_metrics(y_true, y_pred_proba, k=10):
    """Compute Precision@K and NDCG@K"""
    n = len(y_true)

    # Get top-K predictions
    top_k_idx = np.argsort(y_pred_proba)[-k:]  #  This gets HIGHEST scores

    # Precision@K: % of top-K that are actually positive
    precision_k = y_true.iloc[top_k_idx].mean()  #  All are positive!

In [22]:
from sklearn.metrics import ndcg_score, precision_recall_curve

print("=== Ranking Evaluation ===\n")

# Check class balance first
print(f"Test set positive rate: {y_test.mean()*100:.1f}%")
print(f"Random baseline Precision@10: {y_test.mean():.4f}")

# Method 1: Global Precision@K and NDCG@K
def compute_global_ranking_metrics(y_true, y_pred_proba, k=10):
    """Compute global ranking metrics"""
    n = len(y_true)

    # Get top-K predictions (highest scores)
    top_k_idx = np.argsort(y_pred_proba)[-k:]

    # Precision@K: % of top-K that are actually positive
    precision_k = y_true.iloc[top_k_idx].mean()

    # NDCG@K
    y_true_array = np.array([y_true.values])
    y_pred_array = np.array([y_pred_proba])
    ndcg_k = ndcg_score(y_true_array, y_pred_array, k=k)

    return precision_k, ndcg_k

precision_10, ndcg_10 = compute_global_ranking_metrics(
    y_test.reset_index(drop=True),
    y_proba_xgb,
    k=10
)

print(f"\n--- Global Ranking Metrics ---")
print(f"Precision@10: {precision_10:.4f}")
print(f"NDCG@10: {ndcg_10:.4f}")

#  Check if this is just reflecting class imbalance
if abs(precision_10 - y_test.mean()) < 0.05:
    print(" Warning: Precision@10 ≈ baseline (possible leakage or no ranking signal)")

# Method 2: Per-User Ranking (More realistic)
print("\n--- Per-User Ranking Evaluation ---")

# Add user_id back to test set for grouping
test_df_eval = test_df[['customer_id']].reset_index(drop=True)
test_df_eval['y_true'] = y_test.values
test_df_eval['y_pred'] = y_proba_xgb

# Filter users with at least 10 reviews in test set
user_review_counts = test_df_eval.groupby('customer_id').size()
valid_users = user_review_counts[user_review_counts >= 10].index

test_df_eval_filtered = test_df_eval[test_df_eval['customer_id'].isin(valid_users)]

print(f"Evaluating {len(valid_users)} users with ≥10 test reviews")

# Compute per-user Precision@K
def compute_per_user_precision(group, k=10):
    """Compute Precision@K for a single user"""
    if len(group) < k:
        return None

    # Sort by predicted score (descending)
    sorted_group = group.sort_values('y_pred', ascending=False)

    # Get top-K
    top_k = sorted_group.head(k)

    # Precision@K
    return top_k['y_true'].mean()

per_user_precision = test_df_eval_filtered.groupby('customer_id').apply(
    lambda g: compute_per_user_precision(g, k=5)  # Use k=5 for users with fewer reviews
).dropna()

print(f"\nPer-User Precision@5:")
print(f"  Mean: {per_user_precision.mean():.4f}")
print(f"  Median: {per_user_precision.median():.4f}")
print(f"  Std: {per_user_precision.std():.4f}")
print(f"  Min: {per_user_precision.min():.4f}")
print(f"  Max: {per_user_precision.max():.4f}")

# Distribution of per-user precision
print(f"\nDistribution of per-user Precision@5:")
print(per_user_precision.value_counts(bins=5, sort=False).sort_index())

# Method 3: Check if model is actually discriminating
print("\n--- Model Discrimination Check ---")

# AUC (already computed)
print(f"AUC: {auc_xgb:.4f}")

# Brier Score (calibration)
from sklearn.metrics import brier_score_loss
brier = brier_score_loss(y_test, y_proba_xgb)
print(f"Brier Score: {brier:.4f} (lower is better)")

# Prediction distribution
print(f"\nPrediction score distribution:")
print(f"  Mean: {y_proba_xgb.mean():.4f}")
print(f"  Median: {np.median(y_proba_xgb):.4f}")
print(f"  Std: {y_proba_xgb.std():.4f}")
print(f"  Min: {y_proba_xgb.min():.4f}")
print(f"  Max: {y_proba_xgb.max():.4f}")

# Check if predictions are well-separated
pos_scores = y_proba_xgb[y_test == 1]
neg_scores = y_proba_xgb[y_test == 0]

print(f"\nScore separation:")
print(f"  Positive class mean: {pos_scores.mean():.4f}")
print(f"  Negative class mean: {neg_scores.mean():.4f}")
print(f"  Separation: {pos_scores.mean() - neg_scores.mean():.4f}")

# Method 4: Visualize ranking quality
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Score distribution by class
axes[0].hist(neg_scores, bins=30, alpha=0.6, label='Negative (rating <4)', color='coral')
axes[0].hist(pos_scores, bins=30, alpha=0.6, label='Positive (rating ≥4)', color='steelblue')
axes[0].set_xlabel('Predicted Probability', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Prediction Score Distribution by True Label', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Precision-Recall curve
from sklearn.metrics import precision_recall_curve, average_precision_score

precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, y_proba_xgb)
ap_score = average_precision_score(y_test, y_proba_xgb)

axes[1].plot(recall_curve, precision_curve, linewidth=2, color='navy')
axes[1].axhline(y=y_test.mean(), color='red', linestyle='--',
                label=f'Baseline (random): {y_test.mean():.3f}')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title(f'Precision-Recall Curve (AP={ap_score:.3f})', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage Precision (AP): {ap_score:.4f}")

# Final diagnosis
print("\n" + "="*60)
print("DIAGNOSIS")
print("="*60)

if precision_10 > 0.95 and abs(precision_10 - y_test.mean()) < 0.05:
    print(" HIGH RISK: Precision@10 ≈ class baseline")
    print("   → Model may not be learning meaningful ranking")
    print("   → Check for data leakage in features")
elif auc_xgb > 0.8 and per_user_precision.mean() < 0.7:
    print(" MODERATE: Good AUC but lower per-user precision")
    print("   → Model generalizes but has room for improvement")
elif auc_xgb < 0.65:
    print(" LOW PERFORMANCE: AUC too low")
    print("   → Features may lack predictive power")
else:
    print(" GOOD: Model shows reasonable discrimination")
    print("   → Suitable for ranking applications")

print("="*60)

=== Ranking Evaluation ===

Test set positive rate: 86.3%
Random baseline Precision@10: 0.8626

--- Global Ranking Metrics ---
Precision@10: 1.0000
NDCG@10: 1.0000

--- Per-User Ranking Evaluation ---
Evaluating 595 users with ≥10 test reviews

Per-User Precision@5:
  Mean: 0.9176
  Median: 1.0000
  Std: 0.1621
  Min: 0.0000
  Max: 1.0000

Distribution of per-user Precision@5:
(-0.002, 0.2]      6
(0.2, 0.4]        11
(0.4, 0.6]        42
(0.6, 0.8]       101
(0.8, 1.0]       435
Name: count, dtype: int64

--- Model Discrimination Check ---
AUC: 0.6665
Brier Score: 0.1141 (lower is better)

Prediction score distribution:
  Mean: 0.8752
  Median: 0.8749
  Std: 0.0647
  Min: 0.5564
  Max: 0.9944

Score separation:
  Positive class mean: 0.8803
  Negative class mean: 0.8431
  Separation: 0.0372

Average Precision (AP): 0.9258

DIAGNOSIS
 GOOD: Model shows reasonable discrimination
   → Suitable for ranking applications
